# Data Exploration - Full-Plant Images

This notebook explores the full-plant image dataset for AgriGuard AI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## Load Metadata

In [ ]:
# Load metadata
metadata_path = '../data/raw/metadata.csv'
df = pd.read_csv(metadata_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## Data Distribution Analysis

In [ ]:
# Analyze splits
if 'split' in df.columns:
    split_counts = df['split'].value_counts()
    print("Split distribution:")
    print(split_counts)
    
    plt.figure(figsize=(8, 5))
    split_counts.plot(kind='bar', color=['#2ecc71', '#3498db', '#e74c3c'])
    plt.title('Data Split Distribution')
    plt.xlabel('Split')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
# Analyze crop types
if 'crop_type' in df.columns:
    crop_counts = df['crop_type'].value_counts()
    print("\nCrop type distribution:")
    print(crop_counts)
    
    plt.figure(figsize=(10, 6))
    crop_counts.plot(kind='bar', color='#3498db')
    plt.title('Crop Type Distribution')
    plt.xlabel('Crop Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze health status
if 'health_status' in df.columns:
    health_counts = df['health_status'].value_counts()
    print("\nHealth status distribution:")
    print(health_counts)
    
    plt.figure(figsize=(8, 5))
    colors = {'healthy': '#2ecc71', 'stressed': '#f39c12', 'diseased': '#e74c3c'}
    health_counts.plot(kind='bar', 
                      color=[colors.get(x, '#95a5a6') for x in health_counts.index])
    plt.title('Health Status Distribution')
    plt.xlabel('Health Status')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
# Analyze severity distribution
if 'severity' in df.columns:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    df['severity'].hist(bins=20, color='#3498db', edgecolor='black')
    plt.title('Severity Distribution')
    plt.xlabel('Severity Score')
    plt.ylabel('Count')
    
    plt.subplot(1, 2, 2)
    df.boxplot(column='severity')
    plt.title('Severity Boxplot')
    plt.tight_layout()
    plt.show()

## Disease and Pest Analysis

In [ ]:
# Disease labels
disease_cols = ['early_blight', 'late_blight', 'powdery_mildew', 'leaf_spot', 
               'bacterial_spot', 'viral_infection', 'fungal_infection']
disease_cols = [col for col in disease_cols if col in df.columns]

if disease_cols:
    disease_counts = df[disease_cols].sum().sort_values(ascending=False)
    print("Disease occurrence counts:")
    print(disease_counts)
    
    plt.figure(figsize=(12, 6))
    disease_counts.plot(kind='bar', color='#e74c3c')
    plt.title('Disease Occurrence Counts')
    plt.xlabel('Disease Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Pest labels
pest_cols = ['aphids', 'whiteflies', 'thrips', 'spider_mites', 'caterpillars']
pest_cols = [col for col in pest_cols if col in df.columns]

if pest_cols:
    pest_counts = df[pest_cols].sum().sort_values(ascending=False)
    print("\nPest occurrence counts:")
    print(pest_counts)
    
    plt.figure(figsize=(10, 6))
    pest_counts.plot(kind='bar', color='#27ae60')
    plt.title('Pest Occurrence Counts')
    plt.xlabel('Pest Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Abiotic stress labels
abiotic_cols = ['nutrient_deficiency', 'water_stress', 'heat_stress', 
                'salt_stress', 'light_stress']
abiotic_cols = [col for col in abiotic_cols if col in df.columns]

if abiotic_cols:
    abiotic_counts = df[abiotic_cols].sum().sort_values(ascending=False)
    print("\nAbiotic stress occurrence counts:")
    print(abiotic_counts)
    
    plt.figure(figsize=(10, 6))
    abiotic_counts.plot(kind='bar', color='#9b59b6')
    plt.title('Abiotic Stress Occurrence Counts')
    plt.xlabel('Stress Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Sample Image Visualization

In [ ]:
# Display sample images
def display_sample_images(df, num_samples=6):
    """Display sample images with their labels"""
    sample_df = df.sample(min(num_samples, len(df)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        if idx >= len(axes):
            break
        
        img_path = os.path.join('../data/raw', row['image_path'])
        
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[idx].imshow(img)
            
            # Create title
            title = f"{row.get('crop_type', 'Unknown')}\n"
            title += f"Health: {row.get('health_status', 'Unknown')}\n"
            title += f"Severity: {row.get('severity', 'N/A')}"
            
            axes[idx].set_title(title, fontsize=10)
            axes[idx].axis('off')
        else:
            axes[idx].text(0.5, 0.5, 'Image not found', 
                          ha='center', va='center', transform=axes[idx].transAxes)
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

display_sample_images(df, num_samples=6)

## Correlation Analysis

In [ ]:
# Correlation matrix for labels
label_cols = disease_cols + pest_cols + abiotic_cols
if label_cols:
    correlation_matrix = df[label_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, fmt='.2f', cbar_kws={"shrink": 0.8})
    plt.title('Label Correlation Matrix')
    plt.tight_layout()
    plt.show()

## Class Imbalance Analysis

In [ ]:
# Calculate class imbalance
if label_cols:
    label_counts = df[label_cols].sum()
    total_samples = len(df)
    
    imbalance_df = pd.DataFrame({
        'Label': label_cols,
        'Count': label_counts.values,
        'Percentage': (label_counts.values / total_samples * 100).round(2)
    })
    
    print("Class Imbalance Analysis:")
    print(imbalance_df.to_string(index=False))
    
    plt.figure(figsize=(12, 6))
    plt.bar(imbalance_df['Label'], imbalance_df['Percentage'], color='#3498db')
    plt.title('Class Distribution Percentage')
    plt.xlabel('Label')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
print("Dataset Summary:")
print(f"Total samples: {len(df)}")
print(f"Number of crop types: {df['crop_type'].nunique() if 'crop_type' in df.columns else 'N/A'}")
print(f"Number of disease classes: {len(disease_cols)}")
print(f"Number of pest classes: {len(pest_cols)}")
print(f"Number of abiotic stress classes: {len(abiotic_cols)}")
print(f"\nSeverity statistics:")
if 'severity' in df.columns:
    print(df['severity'].describe())